# Demo 01 - Planning Baseline Lakehouse

This notebook loads a deterministic planning baseline for a fictional integrated test and stores it in a Fabric Lakehouse table.

> All identifiers, sites, measures, and event labels in this notebook are invented for demonstration only. They are not operationally representative.


## Cell 1 - Load the checked-in planning baseline
Upload emulated_test_baseline.csv to the Lakehouse Files area before running the next cell.


In [ ]:
# Read the shared synthetic planning projection from the Lakehouse Files area.
baseline_df = (
    spark.read.option("header", "true")
    .option("inferSchema", "false")
    .csv("Files/shared/integrated-test-data/projections/foundation/emulated_test_baseline.csv")
)

row_count = baseline_df.count()
print(f"Loaded {row_count} planning rows from shared release 2026.11.03.")
baseline_df.show(5, truncate=False)


## Cell 2 - Validate the required planning fields
These checks confirm the file has the canonical identifiers, timestamps, and explicit fictional labels expected by downstream demos.


In [ ]:
from pyspark.sql import functions as F

required_columns = [
    "scenario_id",
    "simulation_run_id",
    "test_event_id",
    "site_id",
    "system_instance_id",
    "event_type",
    "event_time_utc",
    "predicted_value",
    "unit",
    "model_version",
    "test_objective_id",
    "required_feed_id",
    "schema_version",
    "classification",
    "fictional_label",
    "location_notice",
]

missing_columns = [name for name in required_columns if name not in baseline_df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

blank_report = (
    baseline_df.select([
        F.sum(
            F.when(
                F.col(name).isNull() | (F.trim(F.col(name)) == ""),
                1,
            ).otherwise(0)
        ).alias(name)
        for name in required_columns
    ])
    .collect()[0]
    .asDict()
)

blank_fields = {name: count for name, count in blank_report.items() if count}
if blank_fields:
    raise ValueError(f"Required fields contain blank values: {blank_fields}")

timestamp_failures = baseline_df.filter(
    F.to_timestamp("event_time_utc", "yyyy-MM-dd'T'HH:mm:ssX").isNull()
).count()
if timestamp_failures:
    raise ValueError(f"Invalid event_time_utc values: {timestamp_failures}")

classification_values = {
    row["classification"] for row in baseline_df.select("classification").distinct().collect()
}
if classification_values != {"SYNTHETIC UNCLASS"}:
    raise ValueError(f"Unexpected classification values: {sorted(classification_values)}")

fictional_labels = {
    row["fictional_label"] for row in baseline_df.select("fictional_label").distinct().collect()
}
if fictional_labels != {"FICTIONAL / DEMO ONLY"}:
    raise ValueError(f"Unexpected fictional labels: {sorted(fictional_labels)}")

print("Required planning fields, timestamps, and fictional labels are valid.")
baseline_df.select(required_columns).show(5, truncate=False)


## Cell 3 - Create the mission_emulated_baseline table
This overwrite keeps the demo idempotent, so re-running the notebook refreshes the same planning baseline.


In [ ]:
(
    baseline_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mission_emulated_baseline")
)

print("Created or replaced the mission_emulated_baseline table.")
print(f"Lakehouse row count: {spark.table('mission_emulated_baseline').count()}")


## Cell 4 - Review objective and required feed coverage
Use this query to see which fictional objectives depend on which planned data feeds.


In [ ]:
%%sql
SELECT
    scenario_id,
    test_objective_id,
    required_feed_id,
    COUNT(*) AS planned_records,
    MIN(model_version) AS model_version
FROM mission_emulated_baseline
GROUP BY scenario_id, test_objective_id, required_feed_id
ORDER BY test_objective_id, required_feed_id


## Cell 5 - Inspect predicted events for downstream handoff
This query orders the invented events exactly as later setup and observed-data demos will expect them.


In [ ]:
%%sql
SELECT
    event_time_utc,
    site_id,
    system_instance_id,
    event_type,
    predicted_value,
    unit,
    test_objective_id,
    required_feed_id,
    model_version
FROM mission_emulated_baseline
ORDER BY event_time_utc, site_id, system_instance_id
